In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure plots are displayed inline
%matplotlib inline

## 1. Define Geometric Brownian Motion (GBM)

The formula for GBM is:
$$ dS_t = \mu S_t dt + \sigma S_t dW_t $$

Where:
- $S_t$ is the price at time $t$
- $\mu$ is the drift (expected return)
- $\sigma$ is the volatility
- $dW_t$ is a Wiener process (random shock)

In [ ]:
def generate_gbm_data(start_price=100, mu=0.05, sigma=0.2, days=1000, dt=1/252):
    """
    Generates a price path using Geometric Brownian Motion.
    """
    n_steps = int(days)
    t = np.linspace(0, days * dt, n_steps)
    
    # Brownian Motion
    W = np.random.standard_normal(size=n_steps)
    W = np.cumsum(W) * np.sqrt(dt)
    
    # Geometric Brownian Motion
    X = (mu - 0.5 * sigma**2) * t + sigma * W
    S = start_price * np.exp(X)
    
    return S

# Generate a sample path
price_path = generate_gbm_data(days=500)
plt.figure(figsize=(10, 6))
plt.plot(price_path)
plt.title("Synthetic Price Path (GBM)")
plt.xlabel("Days")
plt.ylabel("Price")
plt.show()

## 2. Create OHLCV Data

Strategies often need Open, High, Low, Close, and Volume. We can simulate these around the close price.

In [ ]:
def create_synthetic_ohlcv(length=1000, start_price=100, volatility=0.02):
    dates = pd.date_range(start='2020-01-01', periods=length, freq='D')
    
    # Generate Close prices using a random walk for simplicity here
    returns = np.random.normal(0, volatility, length)
    close_prices = start_price * (1 + returns).cumprod()
    
    data = {
        'open': [],
        'high': [],
        'low': [],
        'close': close_prices,
        'volume': []
    }
    
    for close in close_prices:
        # Simulate intraday movement
        daily_range = close * volatility * np.random.uniform(0.5, 1.5)
        
        high = close + (daily_range / 2)
        low = close - (daily_range / 2)
        open_p = np.random.uniform(low, high)
        
        data['open'].append(open_p)
        data['high'].append(high)
        data['low'].append(low)
        data['volume'].append(np.random.randint(1000, 100000))
        
    df = pd.DataFrame(data, index=dates)
    
    # Add 'iv' column for Volatility Arb strategy testing
    # Mean reverting IV around 0.20
    iv = [0.20]
    for _ in range(length-1):
        change = np.random.normal(0, 0.01)
        new_iv = iv[-1] + change + 0.05 * (0.20 - iv[-1]) # Mean reversion
        iv.append(max(0.05, new_iv))
        
    df['iv'] = iv
    
    return df

# Generate and View
df_synthetic = create_synthetic_ohlcv(length=2000)
df_synthetic.tail()

## 3. Save to CSV

Save this data to a file so the `strategy_analyzer.py` can load it.

In [ ]:
import os

# Create data directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Save
file_path = 'data/synthetic_data.csv'
df_synthetic.to_csv(file_path)
print(f"Saved synthetic data to {file_path}")